In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DATASET_FOLDER = "/content/drive/MyDrive/garbage_datasets"

print("📂 Archivos ZIP disponibles en Drive:")
zip_files = [f for f in os.listdir(DRIVE_DATASET_FOLDER) if f.endswith('.zip')]
for i, f in enumerate(zip_files):
    size_mb = os.path.getsize(os.path.join(DRIVE_DATASET_FOLDER, f)) / (1024*1024)
    print(f"{i}: {f} ({size_mb:.2f} MB)")

# Selecciona el archivo (cambia el índice si es diferente)
ZIP_INDEX = 0  # normalmente el más reciente
ZIP_FILENAME = zip_files[ZIP_INDEX]
ZIP_PATH = os.path.join(DRIVE_DATASET_FOLDER, ZIP_FILENAME)
print(f"\n✅ Usando: {ZIP_FILENAME}")

Mounted at /content/drive
📂 Archivos ZIP disponibles en Drive:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/garbage_datasets'

In [ ]:
import zipfile
import os
import shutil
import glob

EXTRACT_BASE = "/content"
TARGET_DIR = os.path.join(EXTRACT_BASE, "yolo_dataset")

# Eliminar carpeta destino si existe
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

print(f"⏳ Descomprimiendo {ZIP_FILENAME}...")
temp_dir = os.path.join(EXTRACT_BASE, "temp_extract")
os.makedirs(temp_dir, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(temp_dir)

print("✅ Descompresión completada. Buscando la carpeta correcta...")

# Inspeccionar el contenido de temp_dir
items = os.listdir(temp_dir)
print(f"Contenido de temp_dir: {items}")

# Función para buscar recursivamente una carpeta que contenga 'images' y 'labels'
def find_dataset_folder(base_dir, target_name="yolo_dataset"):
    for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == target_name:
            # Verificar que contiene images y labels
            if os.path.isdir(os.path.join(root, "images")) and os.path.isdir(os.path.join(root, "labels")):
                return root
    # Si no, buscar cualquier carpeta que contenga images y labels
    for root, dirs, files in os.walk(base_dir):
        if os.path.isdir(os.path.join(root, "images")) and os.path.isdir(os.path.join(root, "labels")):
            return root
    return None

found_path = find_dataset_folder(temp_dir)
if found_path:
    print(f"✅ Estructura de dataset encontrada en: {found_path}")
    # Mover al destino final
    shutil.move(found_path, TARGET_DIR)
    # Eliminar temp_dir (ya no necesitamos)
    shutil.rmtree(temp_dir)
    print(f"✅ Dataset movido a {TARGET_DIR}")
else:
    # Si no encontramos, asumimos que los archivos están directamente en temp_dir
    # y los movemos manualmente
    print("⚠️ No se encontró la carpeta 'yolo_dataset'. Moviendo todo a la carpeta destino...")
    os.makedirs(TARGET_DIR, exist_ok=True)
    for item in os.listdir(temp_dir):
        src = os.path.join(temp_dir, item)
        dst = os.path.join(TARGET_DIR, item)
        shutil.move(src, dst)
    shutil.rmtree(temp_dir)
    print(f"✅ Contenido movido a {TARGET_DIR}")

# Verificación final
if os.path.exists(TARGET_DIR):
    print("📁 Estructura final:")
    for root, dirs, files in os.walk(TARGET_DIR):
        level = root.replace(TARGET_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            for f in files[:3]:
                print(f'{indent}  {f}')
            if len(files) > 3:
                print(f'{indent}  ... y {len(files)-3} más')
else:
    print("❌ ERROR CRÍTICO: No se pudo crear /content/yolo_dataset")

# IMPORTANTE

El orden de las clases no es igual que el de antes, para saber que id corresponde a cada clase ejecutar las siguientes dos celdas

In [1]:
import os
from collections import Counter

DATASET_PATH = "/content/yolo_dataset"
all_ids = []

for split in ['train', 'val', 'test']:
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
    if not os.path.exists(lbl_dir):
        continue
    for file in os.listdir(lbl_dir):
        if file.endswith('.txt'):
            with open(os.path.join(lbl_dir, file), 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        all_ids.append(int(parts[0]))

counter = Counter(all_ids)
print("IDs encontrados en las etiquetas (frecuencia):")
for cid, count in sorted(counter.items()):
    print(f"  ID {cid}: {count} imágenes")

print(f"\nTotal de clases distintas: {len(counter)}")
print(f"IDs presentes: {sorted(counter.keys())}")

IDs encontrados en las etiquetas (frecuencia):

Total de clases distintas: 0
IDs presentes: []


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import random

def show_example_for_id(target_id, split='train'):
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
    img_dir = os.path.join(DATASET_PATH, 'images', split)
    if not os.path.exists(lbl_dir) or not os.path.exists(img_dir):
        print(f"Directorio no encontrado para {split}")
        return

    # Buscar un archivo de etiqueta con ese ID
    for file in os.listdir(lbl_dir):
        if file.endswith('.txt'):
            with open(os.path.join(lbl_dir, file), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if parts and int(parts[0]) == target_id:
                        img_file = file.replace('.txt', '.jpg')
                        img_path = os.path.join(img_dir, img_file)
                        if os.path.exists(img_path):
                            img = cv2.imread(img_path)
                            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            plt.figure(figsize=(4,4))
                            plt.imshow(img_rgb)
                            plt.title(f"ID {target_id}")
                            plt.axis('off')
                            plt.show()
                            return
    print(f"No se encontró imagen con ID {target_id} en {split}")

# Mostrar un ejemplo de cada ID
for cid in sorted(counter.keys()):
    show_example_for_id(cid)
    print("-"*30)

----

In [ ]:
import os

DATASET_PATH = "/content/yolo_dataset"

CLASS_NAMES = ['biological', 'cardboard', 'glass', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)

# Crear dataset.yaml
yaml_content = f"""
path: {DATASET_PATH}
train: images/train
val: images/val
test: images/test

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""

yaml_path = os.path.join(DATASET_PATH, "dataset.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ dataset.yaml guardado en {yaml_path}")
print(f"Clases ({len(CLASS_NAMES)}): {CLASS_NAMES}")

In [ ]:
# Desinstalar versiones conflictivas
!pip uninstall -y ultralytics torch torchvision

# Instalar versiones específicas que sabemos que funcionan juntas
!pip install torch==2.3.0 torchvision==0.18.0
!pip install ultralytics==8.2.0

print("✅ Instalación completada. Reinicia el runtime (Entorno de ejecución → Reiniciar sesión) y luego vuelve a ejecutar desde la celda de montaje de Drive.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
import os

def check_sample(split='train'):
    img_dir = os.path.join(DATASET_PATH, 'images', split)
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
    if not os.path.exists(img_dir):
        print(f"❌ {img_dir} no existe")
        return
    images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    if not images:
        print("No hay imágenes")
        return
    img_file = random.choice(images)
    img_path = os.path.join(img_dir, img_file)
    lbl_path = os.path.join(lbl_dir, img_file.replace('.jpg', '.txt'))
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            line = f.readline().strip().split()
            if line:
                class_id = int(line[0])
                x_center = float(line[1]) * w
                y_center = float(line[2]) * h
                box_w = float(line[3]) * w
                box_h = float(line[4]) * h
                x1 = int(x_center - box_w/2)
                y1 = int(y_center - box_h/2)
                x2 = int(x_center + box_w/2)
                y2 = int(y_center + box_h/2)
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img_rgb, CLASS_NAMES[class_id], (x1, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title(f"{split.upper()} - {img_file}")
    plt.axis('off')
    plt.show()

check_sample('train')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # o 'yolov8n.pt' si quieres más rápido

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    # project y name ahora sin barras
    project='garbage_detector',   # nombre de la carpeta raíz (dentro de runs/)
    name='experiment1',           # subcarpeta para este entrenamiento
    exist_ok=True,               # si ya existe, sobrescribe
    pretrained=True,
    optimizer='auto',
    patience=20,
    amp=True,
)

print("\n✅ Entrenamiento completado.")
# La ruta correcta del mejor modelo será:
# /content/runs/garbage_detector/experiment1/weights/best.pt
best_path = model.trainer.best
print(f"📁 Mejor modelo: {best_path}")

# Probar con test

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import random
import os

best_path = "/content/runs/garbage_detector/weights/best.pt"
if os.path.exists(best_path):
    model_trained = YOLO(best_path)
    test_dir = os.path.join(DATASET_PATH, 'images', 'test')
    if os.path.exists(test_dir):
        test_imgs = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]
        if test_imgs:
            sample = random.choice(test_imgs)
            sample_path = os.path.join(test_dir, sample)
            results = model_trained(sample_path)
            annotated = results[0].plot()
            plt.figure(figsize=(8,8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
            print(results[0].boxes)

# Guardar en Drive

In [ ]:
best_path = "/content/runs/garbage_detector/weights/best.pt"
if os.path.exists(best_path):
    drive_models = "/content/drive/MyDrive/garbage_datasets/models"
    os.makedirs(drive_models, exist_ok=True)
    dest = os.path.join(drive_models, "yolo_best_model.pt")
    shutil.copy2(best_path, dest)
    print(f"✅ Modelo guardado en {dest}")
else:
    print("⚠️ Modelo no encontrado.")